# Testing scientific software: from unit tests to CI/CD

Design evidence at multiple scales—from one numerical function to a cross-platform automated
workflow—and learn what each form of testing can and cannot establish.

**Lecture 3 · Notebook 02 · CMOR 438 / INDE 577**

## Orientation: this is a long-form reference

**Live core:** test intent, unit/integration/system scope, case selection, pytest mechanics,
scientific and ML risks, and this repository's CI workflow.

**Practice:** classify tests, design data-split checks, read failures, and write a focused package
test.

**Extension:** property-based testing, test doubles, mutation testing, and a delivery/deployment
design exercise.

Testing is not a single lecture topic we finish today. These patterns recur throughout the course.

## How to use this notebook

**Estimated time:** 120 minutes core, plus 90 minutes of practice and extension.

**Prerequisites:** functions, exceptions, classes, native files, package imports, subprocesses, and
the completed Rice DSM environment.

Run the core route top to bottom. Read every test name as a sentence about expected behavior. Before
running an example, identify its **risk**, **system under test**, **input**, **oracle**, and **failure
message**. Commands are cross-platform and operate only on this repository or temporary files.

## Learning objectives

By the end, you should be able to:

- explain verification, validation, test oracle, fixture, test double, regression, and flakiness;
- distinguish unit, component, integration, contract, system, end-to-end, smoke, and acceptance tests;
- choose typical, boundary, invalid, structural, and adversarial cases from a contract;
- use assertions, `pytest.raises`, parametrization, fixtures, temporary paths, monkeypatching, and
  output capture appropriately;
- compare example-based, property-based, metamorphic, differential, snapshot, and doctest strategies;
- test numerical, data, stochastic, and machine-learning behavior without making false guarantees;
- distinguish continuous integration, continuous delivery, and continuous deployment;
- read a GitHub Actions workflow as triggers, jobs, runners, steps, gates, permissions, and artifacts;
- diagnose test and automation failures without automatically blaming either code or test; and
- design a risk-based verification strategy rather than maximizing a test count.

## Why this matters in industry and science

Scientific software can execute successfully and still be wrong: units can be mixed, labels can leak
into training, a metric can silently align unequal sequences, a random split can overlap, or an
optimization can converge to the wrong objective. Tests make selected expectations repeatable, but
passing tests do not prove a scientific claim.

The professional question is not “Do we have tests?” It is: **Which failures matter, what evidence
would expose them, at which boundary, how quickly, and under whose responsibility?**

## Worked example: prediction metrics and the scientific knowledge graph

We will test two real package areas built in this course:

```text
rice_dsm.metrics
    └── small numerical units with algebraic properties

rice_dsm.knowledge_graph
    ├── value objects and graph invariants
    ├── JSON + CSV integration
    └── command-line end-to-end behavior
```

The numerical metric lets us reason deeply about oracles, tolerances, properties, and metamorphic
relations. The graph lets us test boundaries among files, domain objects, algorithms, and a CLI.

## Professional practice

| Data scientist asks | Software engineer asks |
| --- | --- |
| Does the metric represent the scientific cost? | Is its interface and numerical policy specified? |
| Are units, shapes, and labels aligned? | Where are those invariants validated? |
| Could data cross a split boundary? | Which automated test detects leakage? |
| Is variation signal, noise, or nondeterminism? | Is the test deterministic, statistical, or flaky? |
| Does performance hold on important subgroups? | Are slice expectations versioned and reviewable? |
| Can another team reproduce the result? | Does CI rebuild and test every supported platform? |
| Should this model be released? | Which artifact, approval, rollback, and monitoring gates apply? |

Good engineering protects the validity of the analysis; it cannot substitute for it.

In [ ]:
import doctest
import inspect
import json
import os
import random
import subprocess
import sys
from collections.abc import Callable, Iterable, Sequence
from math import isfinite
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import Mock

import pytest

from rice_dsm import (
    load_knowledge_graph,
    mean_absolute_error,
    metrics,
    root_mean_squared_error,
)

## 1. Start with language precise enough to prevent false confidence

**Verification** asks whether the software behaves according to a specification. **Validation** asks
whether the software and model are suitable for the real purpose. A test can verify that MAE is
calculated as documented; it cannot decide whether MAE is the right loss for a medical decision.

A **defect** is a problem in an artifact. A **failure** is observed behavior that violates an
expectation. A defect may remain dormant until particular inputs or environments activate it.

### Every test needs an oracle

An **oracle** decides what result should count as correct. Oracles can come from:

- a hand calculation;
- an invariant or mathematical property;
- an independent implementation;
- a trusted reference dataset;
- an approved snapshot;
- a schema or physical constraint;
- a human acceptance criterion; or
- a previous bug report.

The production function cannot simply certify itself. If expected output is computed using the same
logic, the test can repeat the same mistake twice.

### Assertions, tests, and test suites

- An `assert` makes one expectation executable at a point in a program.
- A **test case** arranges a reproducible situation and evaluates one coherent behavior.
- A **test suite** discovers, isolates, runs, and reports many cases.
- A **test runner** such as pytest supplies collection, fixtures, parametrization, reporting, and
  plugins.

Python assertions can be disabled with optimization and should not replace validation of public
inputs. In tests, pytest rewrites assertions to produce useful failure explanations.

## 2. Anatomy of a focused unit test

The common **Arrange–Act–Assert** shape separates inputs, the behavior under test, and its oracle.
Given–When–Then is a behavior-oriented phrasing of the same reasoning. Comments are optional when
names and spacing already make the phases clear.

In [ ]:
# Arrange: a two-observation prediction problem with a hand-computed oracle.
observed_temperature = [1.0, 3.0]
predicted_temperature = [2.0, 5.0]
expected_mae = (1.0 + 2.0) / 2

# Act.
actual_mae = mean_absolute_error(
    observed_temperature,
    predicted_temperature,
)

# Assert.
assert actual_mae == expected_mae
print("MAE:", actual_mae)

This is a **unit test in spirit** because it exercises one function, uses in-memory values, has no
file/network/process dependency, and finishes immediately. The word “unit” describes isolation and
scope, not a required correspondence to one function or class.

Several assertions are appropriate when they describe one coherent returned object. Split tests
when they have unrelated reasons to fail.

### Test names report behavior

```python
def test_mean_absolute_error_matches_hand_calculation() -> None:
    assert mean_absolute_error([1.0, 3.0], [2.0, 5.0]) == 1.5
```

The name completes “the system should…”. A failure report should tell a teammate what contract was
violated without requiring them to decode `test_case_7`.

## 3. Derive cases from the contract

For aligned, finite, nonempty numerical iterables, useful partitions include:

| Case family | Examples | Risk addressed |
| --- | --- | --- |
| typical | several ordinary residuals | core formula |
| boundary | one item; zero residual | endpoint semantics |
| structural | generators; tuples | iterable contract |
| invalid shape | empty; unequal lengths | undefined alignment |
| invalid value | `NaN`, infinity, string, Boolean | numerical domain |
| scale | very large or very small finite values | overflow and precision |

Random examples are not a substitute for reading the domain contract.

In [ ]:
assert mean_absolute_error([4.0], [4.0]) == 0.0
assert mean_absolute_error((value for value in [1.0, 2.0]), [2.0, 4.0]) == 1.5

with pytest.raises(ValueError, match="equal lengths"):
    mean_absolute_error([1.0], [1.0, 2.0])

with pytest.raises(ValueError, match=r"observed\[0\] must be finite"):
    mean_absolute_error([float("nan")], [0.0])

with pytest.raises(TypeError, match=r"observed\[0\] must be a real number"):
    mean_absolute_error([True], [0.0])

### Exceptions are part of the observable interface

Check the narrowest meaningful exception and a stable, actionable part of its message. Testing only
`Exception` can pass when a completely unrelated defect occurs. Testing an entire traceback or exact
path makes the case fragile without adding contract value.

Do not test a private helper merely because it is easier. Reach it through public behavior unless
the helper itself has become a supported interface.

### Exact equality versus numerical tolerance

Use exact equality for exact integers, strings, identifiers, shapes, and results that are specified
to be exact. Use an explicit tolerance when floating-point rounding or an approximate algorithm is
part of the contract.

`pytest.approx` combines relative and absolute tolerances. Choose tolerances from numerical analysis,
measurement precision, and application cost—not merely a value large enough to make a test pass.

In [ ]:
rmse = root_mean_squared_error([0.0, 0.0], [3.0, 4.0])
expected_rmse = 3.5355339059327378

assert rmse == pytest.approx(expected_rmse, rel=1e-12, abs=1e-15)
assert isfinite(rmse)

## 4. Test categories form several dimensions—not one ladder

Teams use overlapping vocabulary. Ask what a label communicates:

| Category | Primary dimension | Purpose |
| --- | --- | --- |
| unit | scope | isolate a small behavior |
| component | scope | exercise a cohesive subsystem |
| integration | boundary | verify components collaborate |
| contract | interface | enforce expectations between producer and consumer |
| system/end-to-end | scope | traverse the assembled application |
| smoke | selection | cheaply detect catastrophic failure |
| acceptance | stakeholder intent | demonstrate agreed user/business behavior |
| regression | history/purpose | prevent a known defect from returning |
| performance | quality attribute | measure time, memory, throughput, or latency |
| security | risk | expose misuse, vulnerabilities, or unsafe boundaries |

One test can be both integration and regression. These are not mutually exclusive species.

### Unit test: numerical behavior in memory

Unit tests should be fast, deterministic, isolated, and precise enough that failure localizes a
problem. They are ideal for calculations, parsers, validators, and domain objects. They cannot reveal
every mismatch between components.

In [ ]:
def test_mae_is_zero_for_perfect_predictions() -> None:
    """A local illustration; pytest tests belong in test files."""

    values = [-2.0, 0.0, 8.5]
    assert mean_absolute_error(values, values) == 0.0


test_mae_is_zero_for_perfect_predictions()

### Integration test: JSON, CSV, adapters, and graph invariants

The graph loader deliberately crosses boundaries: filesystem → parser → mappings → domain objects →
aggregate. A unit test for each converter cannot prove the composed path works with real files.

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the nearest Rice DSM project directory.

    Parameters
    ----------
    start : Path
        Directory from which to search upward.

    Returns
    -------
    Path
        Directory containing this course's project metadata.

    Raises
    ------
    FileNotFoundError
        If the project cannot be found.
    """

    resolved_start = start.resolve()
    for candidate in (resolved_start, *resolved_start.parents):
        project_file = candidate / "pyproject.toml"
        if project_file.is_file() and "rice-dsm" in project_file.read_text(
            encoding="utf-8"
        ):
            return candidate
    raise FileNotFoundError(f"rice-dsm project not found above {resolved_start}")


project_root = find_project_root(Path.cwd())
data_directory = (
    project_root / "notebooks" / "lecture-02-python-foundations-ii" / "data"
)
concepts_path = data_directory / "scientific_concepts.json"
relationships_path = data_directory / "scientific_relationships.csv"

integration_graph = load_knowledge_graph(concepts_path, relationships_path)
assert len(integration_graph) == 20
assert len(integration_graph.relationships) == 25

### System/end-to-end test: enter through the application boundary

An end-to-end test invokes the installed package the way a user does, traversing argument parsing,
files, the domain graph, path search, formatting, stdout, and process status. It offers broad
confidence but a larger failure surface and slower diagnosis.

In [ ]:
end_to_end_process = subprocess.run(
    [
        sys.executable,
        "-m",
        "rice_dsm",
        str(concepts_path),
        str(relationships_path),
        "--path",
        "diffusion_model",
        "scientific_measurement",
    ],
    check=False,
    capture_output=True,
    text=True,
)

assert end_to_end_process.returncode == 0
assert end_to_end_process.stderr == ""
assert end_to_end_process.stdout.startswith("Diffusion model -> Markov chain")

### There is no universal test pyramid

The familiar pyramid—many unit tests, fewer integration tests, few end-to-end tests—is a useful cost
heuristic, not a law. Data pipelines, distributed systems, and user-interface applications may need
more boundary-focused testing. Optimize the portfolio for feedback speed, defect localization,
realism, maintenance cost, and actual risk.

A thousand trivial unit tests do not compensate for one untested data-contract boundary.

## 5. Pytest mechanics that improve test design

Pytest discovers `test_*.py` files and `test_*` functions by convention. Collection should be cheap:
importing a test module must not train a model, download data, or mutate production state.

Useful commands from the repository root:

```text
uv run pytest -q
uv run pytest tests/test_metrics.py -v
uv run pytest tests/test_metrics.py::test_mean_absolute_error_matches_hand_calculation
uv run pytest -k "knowledge_graph and not cli"
uv run pytest -x
uv run pytest --maxfail=3
```

### Parametrization: many cases, one behavior

Parametrization gives every case an independent pytest identity and failure report. Use it when the
same behavior and assertion apply to multiple inputs; do not hide unrelated scenarios inside a giant
table.

```python
@pytest.mark.parametrize("invalid", [float("nan"), float("inf")])
def test_mae_rejects_nonfinite_observations(invalid: float) -> None:
    with pytest.raises(ValueError, match="must be finite"):
        mean_absolute_error([invalid], [0.0])
```

### Fixtures: explicit dependencies with cleanup

A fixture provides a test dependency such as a graph, temporary database, configuration, or client.
Pytest injects it by parameter name. Fixtures can yield and then clean up, but built-ins such as
`tmp_path`, `monkeypatch`, and `capsys` already solve many common needs.

Prefer the narrowest useful scope. A session-scoped mutable fixture can let one test contaminate
another. Avoid “fixture soup” where readers cannot see how a case is arranged.

### `tmp_path`: real filesystem behavior without shared residue

```python
def test_loader_reports_malformed_json(tmp_path: Path) -> None:
    concepts = tmp_path / "concepts.json"
    concepts.write_text("{not-json}", encoding="utf-8")

    with pytest.raises(json.JSONDecodeError):
        load_knowledge_graph(concepts, relationships_path)
```

Temporary paths test genuine file APIs while isolating cases and avoiding personal directories.

In [ ]:
with TemporaryDirectory() as temporary_directory:
    malformed_path = Path(temporary_directory) / "concepts.json"
    malformed_path.write_text("{not-json}", encoding="utf-8")
    with pytest.raises(json.JSONDecodeError):
        load_knowledge_graph(malformed_path, relationships_path)

### `monkeypatch`: replace process state temporarily

Use monkeypatching for environment variables, attributes, dictionaries, current directories, or
network functions at a boundary. Pytest restores changes after the test. Patch the name **where the
system under test looks it up**, not merely where the object was originally defined.

In [ ]:
environment_key = "RICE_DSM_EXPERIMENT_SEED"
assert os.environ.get(environment_key) is None

with pytest.MonkeyPatch.context() as patch:
    patch.setenv(environment_key, "438")
    assert os.environ[environment_key] == "438"

assert os.environ.get(environment_key) is None

### `capsys`, `caplog`, and warnings make channels observable

- `capsys` captures stdout/stderr for functions called inside pytest.
- `caplog` captures log records and levels.
- `pytest.warns` checks intentional warnings.

Prefer returned values for reusable computation. Capture presentation channels when those channels
are the interface—for example, a CLI error must use stderr and a nonzero status.

### Markers, skip, and expected failure

- `skip` means a case is intentionally not run.
- `skipif` encodes an explicit environmental condition.
- `xfail` records a known expected failure and can report an unexpected pass.
- custom markers group categories such as `slow` or `integration` and should be registered.

A permanent skip is hidden risk. Every skip or xfail needs a reason and ownership. Do not use xfail
to normalize a flaky test.

## 6. Test doubles: fake, stub, spy, and mock

Terminology varies, but these distinctions help:

| Double | Typical role |
| --- | --- |
| stub | returns a controlled response |
| fake | working simplified implementation, such as an in-memory store |
| spy | records calls for later assertions |
| mock | configured expectations about interactions |

Use a double when the real collaborator is slow, nondeterministic, destructive, unavailable, or
difficult to control. Do not mock simple value objects or your entire internal implementation.
Over-mocking can prove only that code calls itself in the arrangement the test dictated.

In [ ]:
def publish_alert_if_large_error(
    observed: Iterable[float],
    predicted: Iterable[float],
    *,
    threshold: float,
    publish: Callable[[str], None],
) -> bool:
    """Publish an alert when MAE exceeds a threshold.

    Parameters
    ----------
    observed, predicted : iterable of float
        Aligned finite values passed to ``mean_absolute_error``.
    threshold : float
        Nonnegative MAE threshold in the same unit as the values.
    publish : callable
        Boundary accepting one alert message.

    Returns
    -------
    bool
        Whether an alert was published.

    Raises
    ------
    ValueError
        If ``threshold`` is negative.
    """

    if threshold < 0:
        raise ValueError("threshold must be nonnegative")
    error = mean_absolute_error(observed, predicted)
    if error <= threshold:
        return False
    publish(f"MAE {error:.3f} exceeded threshold {threshold:.3f}")
    return True


publisher = Mock()
did_publish = publish_alert_if_large_error(
    [0.0, 0.0],
    [2.0, 2.0],
    threshold=1.0,
    publish=publisher,
)

assert did_publish is True
publisher.assert_called_once_with("MAE 2.000 exceeded threshold 1.000")

The interaction assertion matters because publishing is the function's external effect. For the MAE
calculation itself, value assertions are stronger and less coupled than mocking internal calls.
Where possible, pass dependencies explicitly instead of patching hidden globals.

## 7. Doctests: executable documentation

Python's `doctest` finds docstring text resembling an interactive session and checks displayed output
exactly. It is well suited to short, deterministic, copyable examples of a public programming
interface (API). It is poor for extensive
case matrices, unstable representations, random output, timestamps, or approximate numerical text.

Doctests complement—not replace—unit tests. A beautiful example and a rigorous boundary suite serve
different readers and risks.

In [ ]:
print(inspect.getdoc(mean_absolute_error))

doctest_results = doctest.testmod(metrics, verbose=False)
print(doctest_results)

assert doctest_results.failed == 0
assert doctest_results.attempted >= 3

### Doctest failure modes

Doctest compares textual representations. Whitespace, dictionary ordering assumptions, platform
paths, object addresses, and floating-point formatting can make examples brittle. Options such as
`ELLIPSIS` and `NORMALIZE_WHITESPACE` exist, but a broad ellipsis can also hide meaningful errors.

Our CI reaches the metric doctests through `tests/test_metrics.py`; documentation examples that no
automation executes will eventually drift.

## 8. Properties and metamorphic relations address weak oracles

Example-based tests ask about selected points. **Property-based testing** generates many values and
checks an invariant. Useful MAE properties include:

- nonnegativity;
- identity: `MAE(y, y) = 0`;
- symmetry: `MAE(y, p) = MAE(p, y)`; and
- translation invariance: adding the same constant to both inputs changes nothing.

A property can be wrong or incomplete. Generators must respect and challenge the intended domain.

In [ ]:
property_random = random.Random(577)
for _ in range(100):
    observed = [property_random.uniform(-100.0, 100.0) for _ in range(8)]
    predicted = [property_random.uniform(-100.0, 100.0) for _ in range(8)]
    error = mean_absolute_error(observed, predicted)

    assert error >= 0.0
    assert error == mean_absolute_error(predicted, observed)
    assert mean_absolute_error(observed, observed) == 0.0

The seeded loop is deterministic teaching code, not a full property-testing engine. Libraries such as
Hypothesis generate and shrink cases so a failure is reduced to a small counterexample. Property
testing still needs hand-selected boundaries and integration tests.

**Metamorphic testing** is especially useful when exact output is hard to know: transform the input
in a way that implies a relation among outputs.

In [ ]:
observed = [-2.0, 1.0, 8.0]
predicted = [0.0, 4.0, 7.0]
baseline_error = mean_absolute_error(observed, predicted)
offset = 10_000.0

translated_error = mean_absolute_error(
    [value + offset for value in observed],
    [value + offset for value in predicted],
)

assert translated_error == baseline_error

### Differential testing, fuzzing, and mutation testing

- **Differential testing** compares independent implementations on shared inputs. Agreement is useful
  evidence, but shared assumptions can produce shared errors.
- **Fuzzing** sends generated, malformed, or adversarial input through an interface to expose crashes,
  hangs, and unsafe parsing.
- **Mutation testing** deliberately changes production code—such as replacing `<` with `<=`—and asks
  whether tests fail. Surviving mutations reveal weak detection, not automatically missing line
  coverage.

These techniques test the tests as much as the implementation.

## 9. Snapshot and golden-file tests

A snapshot stores approved output and reports differences later. It helps with large structured
results, serialized schemas, reports, or UI rendering. Risks include noisy diffs, platform-dependent
content, and “approve all” review that blesses unintended changes.

Prefer the smallest stable semantic snapshot. Normalize timestamps and paths only under an explicit
policy; never erase scientifically meaningful variation merely to stabilize a test.

In [ ]:
graph_snapshot = integration_graph.to_mapping()
stable_summary = {
    "schema_version": graph_snapshot["schema_version"],
    "node_count": len(graph_snapshot["nodes"]),
    "relationship_count": len(graph_snapshot["relationships"]),
    "first_node_id": graph_snapshot["nodes"][0]["identifier"],
}
expected_summary = {
    "schema_version": 1,
    "node_count": 20,
    "relationship_count": 25,
    "first_node_id": "vector",
}

assert stable_summary == expected_summary

## 10. Regression tests and red–green–refactor

A regression test preserves evidence for a defect that actually occurred:

1. reproduce the failure with the smallest representative case;
2. write a test and observe the expected red state;
3. repair the implementation;
4. observe the focused test turn green;
5. run the full suite; and
6. refactor while behavior remains protected.

Observing red matters. A test that passed before the repair may not detect the defect.

In [ ]:
def deliberately_broken_mae(
    observed: Sequence[float],
    predicted: Sequence[float],
) -> float:
    """Demonstrate a defect: divide by one more than the sample count."""

    total = sum(
        abs(actual - estimate)
        for actual, estimate in zip(observed, predicted, strict=True)
    )
    return total / (len(observed) + 1)


try:
    assert deliberately_broken_mae([1.0, 3.0], [2.0, 5.0]) == 1.5
except AssertionError:
    print("Red observed: the hand-calculated regression case detects the defect.")
else:
    raise AssertionError("the demonstration test should detect the deliberate defect")

### Test-driven development is a design loop, not a ritual

TDD can clarify interfaces by writing a caller's expectation first. It is especially effective for
well-understood incremental behavior. Exploratory science may require learning what interface is
appropriate before stabilizing it. Spikes can be discarded or refactored, then protected once the
behavior and scientific meaning are understood.

### Coverage is a question generator, not a quality score

Line and branch coverage identify code not executed by tests. High coverage cannot tell whether
assertions are meaningful, oracles are independent, units are correct, leakage is absent, or the
scientific model is valid. Low coverage can reveal obvious gaps.

Use coverage to ask “Why is this branch untested?” Do not incentivize a percentage in isolation;
people will optimize the number rather than the risk.

## 11. Tests specific to data science and machine learning

Model code sits inside a larger data system. Test layers should include:

| Target | Example contract |
| --- | --- |
| raw ingestion | source fields and provenance retained |
| schema | required columns, types, and unique identifiers |
| semantic values | units, ranges, allowed categories, missingness policy |
| split integrity | entities and future information do not cross boundaries |
| transformations | fitted only on training data; shape and feature order preserved |
| metrics | direction, unit, averaging, labels, and edge behavior documented |
| stochastic training | seeds recorded; variation evaluated statistically |
| model behavior | baseline, slices, calibration, and failure cases monitored |
| serialization | predictions agree before and after round trip within policy |
| serving | request/response schema, latency, and fallback behavior |

Passing model tests does not mean the model is ethical, causal, fair, calibrated, or useful.

### Data tests: schema validity is not scientific validity

A CSV may parse and have the expected columns while recording centimeters under a field documented
as meters. Test structural contracts and semantic contracts separately. Preserve raw data and make
normalization, imputation, filtering, and deduplication observable.

In [ ]:
concept_document = json.loads(concepts_path.read_text(encoding="utf-8"))
concept_records = concept_document["concepts"]
concept_ids = [record["id"] for record in concept_records]

assert concept_document["schema_version"] == 1
required_fields = {"id", "label", "category", "description"}
assert all(required_fields <= record.keys() for record in concept_records)
assert len(concept_ids) == len(set(concept_ids))
assert all(identifier.isidentifier() for identifier in concept_ids)

### Leakage tests protect experimental boundaries

Split integrity should use stable entity or time identifiers, not merely compare row counts. Duplicate
patients, devices, locations, or future observations can leak information even when row indices are
different.

In [ ]:
def assert_disjoint_identifiers(
    training_ids: Iterable[str],
    evaluation_ids: Iterable[str],
) -> None:
    """Raise when an entity appears in training and evaluation data.

    Parameters
    ----------
    training_ids, evaluation_ids : iterable of str
        Stable entity identifiers assigned to each split.

    Raises
    ------
    ValueError
        If either input contains duplicates or the splits overlap.
    """

    training = tuple(training_ids)
    evaluation = tuple(evaluation_ids)
    if len(training) != len(set(training)):
        raise ValueError("training identifiers contain duplicates")
    if len(evaluation) != len(set(evaluation)):
        raise ValueError("evaluation identifiers contain duplicates")
    overlap = sorted(set(training) & set(evaluation))
    if overlap:
        raise ValueError(f"entity leakage across splits: {overlap}")


assert_disjoint_identifiers(["sample-001", "sample-002"], ["sample-003"])
with pytest.raises(ValueError, match="entity leakage"):
    assert_disjoint_identifiers(["sample-001"], ["sample-001"])

### Stochastic and statistical tests require another kind of oracle

Avoid asserting one exact random sample or one model score unless the implementation promises it.
Possible strategies include:

- inject a random generator or seed;
- test deterministic invariants of every sample;
- compare aggregate behavior across many repetitions;
- use a statistically justified tolerance and control false positives;
- separate a fast deterministic suite from slower statistical checks; and
- record enough state to reproduce a discovered counterexample.

A probabilistic test that fails 1% of correct runs will eventually poison trust in CI.

### Model-quality thresholds are acceptance policy, not universal truth

A test such as `assert validation_auc >= 0.80` depends on the dataset version, split, seed, uncertainty,
baseline, subgroup behavior, and cost of errors. A single aggregate threshold can hide degradation in
an important slice.

Separate software invariants (“probabilities are finite and shaped correctly”) from empirical model
claims (“performance meets an approved criterion on versioned evaluation data”).

## 12. Test isolation, determinism, and flakiness

A test should not depend on order, a developer's home directory, network availability, local time,
or residue from another test unless that dependency is its explicit subject. Isolation enables
parallel execution and reliable diagnosis.

A **flaky test** sometimes passes and sometimes fails without a relevant code change. Common causes
include uncontrolled randomness, races, timing assumptions, external services, floating-point
thresholds, shared mutable state, and order dependence.

### Flaky-test response

1. preserve logs, seed, platform, timing, and failing input;
2. reproduce or increase the failure rate;
3. identify the uncontrolled dependency;
4. repair product or test according to the contract;
5. quarantine only when necessary, with ownership and a deadline; and
6. never solve flakiness by blindly retrying until green.

Retries can be appropriate for explicitly transient distributed operations, but they change the
contract and must remain visible.

### Performance tests and benchmarks

A benchmark measures a distribution of time, memory, or throughput under controlled conditions.
Warm-up, machine contention, input scale, and variance matter. A microsecond threshold in ordinary
CI is often flaky.

Use correctness tests for semantics; use benchmark tooling and stable environments for performance.
A broad timeout may protect CI from a hang, but it is not a trustworthy performance claim.

## 13. Run focused tests locally, then the full suite

Fast feedback encourages small changes. During development, run the narrowest relevant case. Before
completion, run broader regression tests and the same checks CI will execute.

The next cell launches only the metrics test module. It skips itself when the entire notebook is
already being executed by the repository's notebook test, preventing nested test recursion.

In [ ]:
if os.environ.get("RICE_DSM_RUNNING_NOTEBOOK_TESTS"):
    print("Nested pytest demonstration skipped during notebook-suite execution.")
else:
    focused_test_process = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            "tests/test_metrics.py",
            "-q",
        ],
        cwd=project_root,
        check=False,
        capture_output=True,
        text=True,
    )
    print(focused_test_process.stdout)
    if focused_test_process.stderr:
        print(focused_test_process.stderr)
    assert focused_test_process.returncode == 0

## 14. Continuous integration is a practice, not a test type

**Continuous integration (CI)** means integrating small changes frequently and automatically
building and checking each shared change. CI runs many verification activities:

- dependency synchronization;
- static analysis and linting;
- package build;
- unit, integration, repository, CLI, doctest, and notebook execution tests; and
- cross-platform compatibility checks.

CI does not replace local testing or review. It supplies a repeatable clean environment and shared
evidence attached to a commit.

### GitHub Actions vocabulary

```text
event → workflow run → job(s) on runner(s) → ordered steps → status/artifacts
```

- A **workflow** is versioned YAML under `.github/workflows/`.
- An **event** such as push or pull request triggers it.
- A **job** runs on one runner and contains ordered steps.
- A **matrix** expands a job over configurations such as operating systems.
- An **action** is a reusable step implementation; `run` executes a command.
- A **gate** requires selected statuses before merge, release, or deployment.

In [ ]:
workflow_path = project_root / ".github" / "workflows" / "course-ci.yml"
workflow_text = workflow_path.read_text(encoding="utf-8")

required_triggers = ("push:", "pull_request:", "workflow_dispatch:")
required_platforms = ("ubuntu-latest", "macos-latest", "windows-latest")
required_commands = (
    "uv sync --locked",
    "uv run ruff check src tests scripts",
    "uv build",
    "uv run pytest -q",
)

assert all(trigger in workflow_text for trigger in required_triggers)
assert all(platform in workflow_text for platform in required_platforms)
assert all(command in workflow_text for command in required_commands)
print(workflow_text)

### Read this repository's CI as a risk-control story

1. Trigger on every push and pull request; allow manual runs.
2. Cancel obsolete runs for the same branch through concurrency control.
3. Grant read-only repository contents permission.
4. expand the quality job across Ubuntu, macOS, and Windows;
5. install pinned tooling and the requested Python;
6. reproduce the locked environment;
7. register and verify the notebook kernel;
8. lint, build the distribution, and run all tests; and
9. require every matrix platform through one aggregate gate.

The matrix catches path, shell, wheel, encoding, and OS behavior that one laptop cannot establish.

### Automation correctness and security

CI configuration is production code. Review:

- least-privilege permissions;
- pinned third-party actions and dependency locks;
- untrusted pull-request code;
- secret exposure in commands, logs, caches, and artifacts;
- cancellation, timeout, and retry policies;
- artifact identity and retention; and
- whether the final gate truly depends on every required job.

A green workflow can be misconfigured to skip the important work. Test the pipeline design, not only
the commands inside it.

### CI failures are evidence, not annoyance

When CI fails but local tests pass, compare Python, lock state, OS, architecture, environment
variables, working directory, file-case rules, line endings, locale, and test order. Do not repeatedly
rerun until one green result appears.

Store useful diagnostics while avoiding secrets and personally identifying data.

## 15. CI, continuous delivery, and continuous deployment

The phrase **CI/CD** compresses distinct practices:

- **Continuous integration:** every integrated change is automatically built and checked.
- **Continuous delivery:** every passing change can produce a releasable artifact; a human or policy
  may decide when to release or deploy it.
- **Continuous deployment:** every passing change automatically reaches production without a manual
  release decision.

Terminology varies across organizations, so define the pipeline behavior rather than relying on the
acronym. This course repository currently has CI; it does not deploy an application.

### A responsible delivery/deployment path

```text
commit → CI checks → immutable artifact → staging verification
       → approval/policy gate → production rollout → monitoring → rollback
```

Build once and promote the same identified artifact. Rebuilding at each stage can produce different
bits. Protect deployment environments, restrict branches, delay or require reviewers when risk
warrants it, scope secrets to the job that needs them, and retain a tested rollback path.

### Deployment tests do not end at release

Pre-deployment checks cannot predict every production interaction. Delivery strategy may include:

- staging or shadow evaluation;
- canary or gradual rollout;
- health checks and smoke tests;
- schema migration compatibility;
- observability for errors, latency, drift, and model-quality proxies;
- explicit rollback triggers; and
- post-incident regression tests.

For ML, monitor data and decision behavior—not only whether the server process is alive.

## 16. Automation extends beyond CI/CD

Useful automation can run:

- on each save or commit: formatting, linting, focused tests;
- on each pull request: full CI, review gates, security checks;
- on a schedule: dependency audits, slow statistical tests, data-drift reports;
- on releases: build, sign, publish, generate changelogs;
- on data arrival: schema validation and pipeline execution; or
- on incidents: rollback, evidence capture, and notification.

Automate a trustworthy decision process, not an opaque sequence of commands. Every automation needs
an owner, failure channel, permissions model, and recovery procedure.

## 17. Design a risk-based test portfolio

Start from failure modes rather than tools:

1. identify a behavior or quality attribute that matters;
2. name the plausible failure and consequence;
3. choose the lowest-cost test boundary capable of detecting it;
4. define an independent oracle and representative cases;
5. decide required isolation, data, platform, and frequency;
6. make failure output actionable;
7. automate at the appropriate stage; and
8. revisit the portfolio after defects and system changes.

Tests have maintenance cost. Delete redundant tests when stronger tests cover the same risk, but never
erase evidence merely to reduce a failing count.

### Example portfolio for the knowledge graph

| Risk | Test boundary | Oracle |
| --- | --- | --- |
| invalid node accepted | unit | identifier/domain contract |
| duplicate triple mutates graph | unit | aggregate invariant |
| CSV header drift | integration | versioned schema |
| missing endpoint enters graph | integration | referential integrity |
| BFS returns a non-shortest route | unit/property | hand graph and path length |
| CLI prints error to stdout | application/contract | channel and exit-status policy |
| import fails outside repository | system | fresh subprocess |
| notebook requires hidden state | system | fresh kernel execution |
| Windows path assumption | CI matrix | same suite on Windows runner |

No one test type covers this portfolio economically.

## 18. Debugging a failed test

A failure means observed behavior differs from encoded expectation. Either side—or the environment—
may be wrong.

1. Read the test name and first relevant assertion.
2. Inspect actual versus expected values and the shortest useful traceback.
3. Reproduce the single case in a clean state.
4. State the intended contract in words and locate its authority.
5. Determine whether product, test, fixture, data, or environment violated it.
6. minimize the counterexample without removing the cause.
7. repair the correct artifact and run focused tests.
8. run broader tests for regressions.

Never weaken an assertion solely to restore green status.

### Common failure modes in testing

| Smell | Consequence | Better question |
| --- | --- | --- |
| only happy paths | boundaries remain unknown | where does valid become invalid? |
| assertion-free test | execution mistaken for correctness | what observable must hold? |
| same formula computes expected | shared defect | what independent oracle exists? |
| exact float comparison everywhere | brittle numerical tests | what tolerance follows from analysis? |
| huge snapshot | unreviewable changes | what minimal semantic output matters? |
| mock every collaborator | tests mirror implementation | which external boundary needs control? |
| shared mutable fixture | order dependence | can each case own its state? |
| sleep-based timing | flaky CI | what event or state can be awaited? |
| rerun failures until green | hides nondeterminism | which dependency is uncontrolled? |
| 100% coverage as goal | metric gaming | which high-consequence risk lacks evidence? |

## Guided practice: classify by scope and purpose

Classify each scenario on at least two dimensions:

1. MAE rejects `NaN`.
2. The JSON/CSV loader rejects an unknown endpoint.
3. `python -m rice_dsm ...` prints a path and returns zero.
4. A former duplicate-node bug cannot return.
5. A deployed service responds to a health endpoint.
6. A domain expert approves a versioned evaluation report.

Success criterion: justify scope, purpose, oracle, cost, and where the case runs. More than one label
may be correct.

In [ ]:
classification = {
    "MAE rejects NaN": ("unit", "domain validation"),
    "loader rejects unknown endpoint": ("integration", "contract"),
    "package CLI prints path": ("end-to-end", "acceptance"),
    "duplicate-node bug stays fixed": ("unit", "regression"),
    "deployed health response": ("system", "smoke"),
    "expert approves evaluation": ("system", "acceptance"),
}

for scenario, labels in classification.items():
    print(f"{scenario:<34} | {', '.join(labels)}")

assert all(len(labels) == 2 for labels in classification.values())

## Independent practice: test split integrity beyond overlap

Extend `assert_disjoint_identifiers` for a time-dependent prediction problem. Specify and test:

- entity disjointness if the scientific design requires it;
- `max(training_time) < min(evaluation_time)`;
- duplicate observation identifiers;
- timezone and missing-time policy; and
- an error message that identifies the violated boundary without exposing sensitive records.

Success criterion: write the contract first, include typical/boundary/invalid cases, and explain which
leakage mechanisms the function still cannot detect.

## Independent practice: add one package test with red–green–refactor

Choose one behavior not yet covered—for example, RMSE symmetry or rejection of Boolean predictions.

1. Write the expected result by hand.
2. Add one focused test to `tests/test_metrics.py`.
3. Run only that node ID and observe whether it is red or already supported.
4. If already green, explain why it still adds distinct risk evidence—or choose another case.
5. Implement only if behavior is missing and desired.
6. Run the full metrics module, then the complete suite.

Success criterion: a reviewer can infer the contract and risk from the test name and assertions.

## Extension: design CI/CD for a scientific model service

Write a staged proposal including:

- pull-request checks and their maximum feedback time;
- unit, data-contract, integration, statistical, security, and performance suites;
- versioned data/model artifacts and provenance;
- protected staging and production environments;
- secrets and least-privilege permissions;
- manual delivery versus automatic deployment decision;
- canary criteria, monitoring, and rollback; and
- ownership when a flaky test, drift alarm, or deployment gate fails.

Defend every gate by the failure it controls. Adding more steps is not automatically safer.

## Retrieval practice

Answer without running code:

1. How do verification and scientific validation differ?
2. What is a test oracle, and why should it be independent?
3. Why can one test be both integration and regression?
4. When should floating-point assertions use a tolerance?
5. What problems do fixtures solve, and how can they create coupling?
6. When is a mock less informative than a real value assertion?
7. What makes a doctest valuable, and what makes it brittle?
8. How do property and metamorphic tests help when exact output is hard to know?
9. Why do coverage and passing tests fail to prove scientific correctness?
10. What ML leakage risks require tests outside model code?
11. How do CI, continuous delivery, and continuous deployment differ?
12. Why should a release promote one immutable artifact?
13. What is the responsible response to a flaky test?

## Takeaway

Testing is structured evidence about risk:

```text
scientific and user intent
    → explicit contracts and failure modes
    → unit + boundary + system evidence
    → automated CI on clean supported environments
    → controlled delivery/deployment
    → production observation and new regression knowledge
```

Choose test techniques for the failures they can reveal. Keep oracles independent, cases isolated,
numerical policies explicit, and automation reviewable. A green pipeline means its encoded checks
passed—not that the software is defect-free or the scientific conclusion is true.

Next, Notebook 03 introduces NumPy arrays with these testing habits already in place.

## Further reading

- [pytest documentation](https://docs.pytest.org/en/stable/)
- [pytest fixtures](https://docs.pytest.org/en/stable/how-to/fixtures.html)
- [pytest parametrization](https://docs.pytest.org/en/stable/how-to/parametrize.html)
- [pytest monkeypatching](https://docs.pytest.org/en/stable/how-to/monkeypatch.html)
- [Python `doctest`](https://docs.python.org/3/library/doctest.html)
- [Python `unittest.mock`](https://docs.python.org/3/library/unittest.mock.html)
- [GitHub Actions: Continuous integration](https://docs.github.com/en/actions/get-started/continuous-integration)
- [GitHub Actions: Workflows](https://docs.github.com/en/actions/concepts/workflows-and-actions/workflows)
- [GitHub Actions: Deployments and environments](https://docs.github.com/en/actions/reference/workflows-and-actions/deployments-and-environments)
- [Google: Rules of Machine Learning](https://developers.google.com/machine-learning/guides/rules-of-ml)